In [21]:

!pip install -q langchain langchain-core langchain-community \
    langchain-google-genai langchain-huggingface \
    langchain-text-splitters faiss-cpu \
    sentence-transformers pypdf \
    google-generativeai

print("✅ All libraries installed successfully!")

✅ All libraries installed successfully!


In [22]:

from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = "/content/drive/MyDrive/Week7_RAG_Project"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/vector_db", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/documents", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/logs", exist_ok=True)

print("✅ Google Drive mounted successfully!")
print(f"✅ Project folder created: {PROJECT_DIR}")
print(f"✅ Vector DB folder: {PROJECT_DIR}/vector_db")
print(f"✅ Documents folder: {PROJECT_DIR}/documents")
print(f"✅ Logs folder: {PROJECT_DIR}/logs")

Mounted at /content/drive
✅ Google Drive mounted successfully!
✅ Project folder created: /content/drive/MyDrive/Week7_RAG_Project
✅ Vector DB folder: /content/drive/MyDrive/Week7_RAG_Project/vector_db
✅ Documents folder: /content/drive/MyDrive/Week7_RAG_Project/documents
✅ Logs folder: /content/drive/MyDrive/Week7_RAG_Project/logs


In [39]:
!pip install langchain-groq -q
print("Done!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.6 MB/s eta 0:00:00
Done!


In [40]:

import os
import warnings
warnings.filterwarnings('ignore')

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from datetime import datetime
import json


os.environ["GROQ_API_KEY"] = "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"

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.3,
    api_key=os.environ["GROQ_API_KEY"]
)


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    cache_folder=f"{PROJECT_DIR}/embeddings_cache"
)

print("✅ All libraries imported!")
print("✅ Groq Llama 3 LLM ready!")
print("✅ Embedding model ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ All libraries imported!
✅ Groq Llama 3 LLM ready!
✅ Embedding model ready!


In [24]:

import urllib.request

PDF_URL = "https://arxiv.org/pdf/1706.03762"
PDF_PATH = f"{PROJECT_DIR}/documents/attention_is_all_you_need.pdf"


if not os.path.exists(PDF_PATH):
    print("Downloading PDF...")
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
    print("✅ PDF downloaded successfully!")
else:
    print("✅ PDF already exists in Drive!")

print(f"📄 PDF saved at: {PDF_PATH}")
print(f"📦 File size: {os.path.getsize(PDF_PATH) / 1024:.2f} KB")

✅ PDF downloaded successfully!
📄 PDF saved at: /content/drive/MyDrive/Week7_RAG_Project/documents/attention_is_all_you_need.pdf
📦 File size: 2163.32 KB


In [25]:

print("Loading PDF...")


loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"✅ PDF loaded successfully!")
print(f"📄 Total pages: {len(pages)}")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " "]
)

chunks = text_splitter.split_documents(pages)

print(f"✅ Total chunks created: {len(chunks)}")
print(f"\n📝 Sample chunk:")
print(chunks[0].page_content[:300])

Loading PDF...
✅ PDF loaded successfully!
📄 Total pages: 15
✅ Total chunks created: 52

📝 Sample chunk:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par


In [33]:

VECTOR_DB_PATH = f"{PROJECT_DIR}/vector_db/faiss_index"


if os.path.exists(f"{VECTOR_DB_PATH}.faiss") or os.path.exists(VECTOR_DB_PATH):
    print("✅ Vector DB already exists — loading from Drive...")
    vectorstore = FAISS.load_local(
        VECTOR_DB_PATH,
        embeddings,
        allow_dangerous_deserialization=True
    )
    print("✅ Vector DB loaded from Drive!")
else:
    print("Creating Vector DB...")
    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings
    )

    vectorstore.save_local(VECTOR_DB_PATH)
    print("✅ Vector DB created and saved to Drive!")

print(f"📊 Total vectors: {vectorstore.index.ntotal}")
print(f"💾 Saved at: {VECTOR_DB_PATH}")

✅ Vector DB already exists — loading from Drive...
✅ Vector DB loaded from Drive!
📊 Total vectors: 52
💾 Saved at: /content/drive/MyDrive/Week7_RAG_Project/vector_db/faiss_index


In [42]:

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)


prompt_template = """I have a research paper and I need help understanding it.
Answer my question using only the content from the paper provided below.
If the answer is not present in the content, just say "I could not find this in the paper."
Try to mention the relevant section or page where you found the answer.

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

def format_docs(docs):
    return "\n\n".join([
        f"[Page {doc.metadata.get('page', 'N/A')}]: {doc.page_content}"
        for doc in docs
    ])


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG Pipeline created successfully!")
print("✅ Retriever ready!")
print("✅ Prompt template ready!")
print("✅ RAG Chain ready!")

✅ RAG Pipeline created successfully!
✅ Retriever ready!
✅ Prompt template ready!
✅ RAG Chain ready!


In [43]:

LOG_FILE = f"{PROJECT_DIR}/logs/qa_logs.json"

def ask_question(question):
    print(f"\n❓ Question: {question}")
    print("-" * 50)


    answer = rag_chain.invoke(question)


    docs = retriever.invoke(question)
    sources = list(set([f"Page {doc.metadata.get('page', 'N/A')}" for doc in docs]))

    print(f"✅ Answer: {answer}")
    print(f"📄 Sources: {sources}")


    log_entry = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "question": question,
        "answer": answer,
        "sources": sources
    }


    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, "r") as f:
            logs = json.load(f)
    else:
        logs = []

    logs.append(log_entry)


    with open(LOG_FILE, "w") as f:
        json.dump(logs, f, indent=4)

    return answer


questions = [
    "What is the main contribution of this paper?",
    "What is the Transformer architecture?",
    "What is multi-head attention?",
    "What datasets were used for training?",
    "What are the results of the model?"
]

for q in questions:
    ask_question(q)
    print("=" * 60)


❓ Question: What is the main contribution of this paper?
--------------------------------------------------
✅ Answer: The main contribution of this paper is the proposal of a new simple network architecture, the Transformer, which is based solely on attention mechanisms, dispensing with recurrence and convolutions. 

[Page 1]: Abstract
📄 Sources: ['Page 2', 'Page 12', 'Page 11', 'Page 0']

❓ Question: What is the Transformer architecture?
--------------------------------------------------
✅ Answer: The Transformer architecture follows the overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder (Page 2, Figure 1). 

The encoder is composed of a stack of N = 6 identical layers, each with two sub-layers: a multi-head self-attention mechanism and a simple, position-wise fully connected feed-forward network (Page 2). 

The decoder also follows a similar architecture, with self-attention layers allowing each position to atten

In [44]:

print("=" * 60)
print("RAG SYSTEM EVALUATION")
print("=" * 60)

eval_questions = [
    {
        "question": "What is the main contribution of this paper?",
        "expected_keywords": ["transformer", "attention", "architecture"]
    },
    {
        "question": "What datasets were used for training?",
        "expected_keywords": ["WMT", "English", "German", "French"]
    },
    {
        "question": "What is multi-head attention?",
        "expected_keywords": ["attention", "heads", "subspaces"]
    }
]

correct = 0
total = len(eval_questions)

for item in eval_questions:
    answer = rag_chain.invoke(item["question"]).lower()
    keywords_found = sum(1 for kw in item["expected_keywords"] if kw.lower() in answer)
    score = keywords_found / len(item["expected_keywords"])

    if score >= 0.5:
        correct += 1
        status = "✅ PASS"
    else:
        status = "❌ FAIL"

    print(f"\nQ: {item['question']}")
    print(f"Keywords found: {keywords_found}/{len(item['expected_keywords'])}")
    print(f"Status: {status}")

accuracy = (correct / total) * 100
print(f"\n{'=' * 60}")
print(f"Overall Accuracy: {accuracy:.2f}%")
print(f"Questions Passed: {correct}/{total}")
print(f"{'=' * 60}")


eval_results = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "total_questions": total,
    "correct": correct,
    "accuracy": accuracy,
    "document": "Attention is All You Need",
    "llm": "Groq Llama 3.1",
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "vector_db": "FAISS"
}

with open(f"{PROJECT_DIR}/logs/evaluation.json", "w") as f:
    json.dump(eval_results, f, indent=4)

print("\n✅ Evaluation saved to Drive!")

RAG SYSTEM EVALUATION

Q: What is the main contribution of this paper?
Keywords found: 3/3
Status: ✅ PASS

Q: What datasets were used for training?
Keywords found: 4/4
Status: ✅ PASS

Q: What is multi-head attention?
Keywords found: 3/3
Status: ✅ PASS

Overall Accuracy: 100.00%
Questions Passed: 3/3

✅ Evaluation saved to Drive!


In [46]:

print("=" * 60)
print("LIVE QUESTION ANSWERING SYSTEM")
print("=" * 60)
print("Type your question and press Enter!")
print("Type 'exit' to stop")
print("=" * 60)

while True:
    question = input("\n❓ Your Question: ")

    if question.lower() == 'exit':
        print("\n👋 Exiting QA System!")
        break

    if not question.strip():
        print("Please enter a valid question!")
        continue

    print("\n🔍 Searching document...")
    answer = ask_question(question)
    print("=" * 60)

LIVE QUESTION ANSWERING SYSTEM
Type your question and press Enter!
Type 'exit' to stop

❓ Your Question: What optimizer was used for training?

🔍 Searching document...

❓ Question: What optimizer was used for training?
--------------------------------------------------
✅ Answer: The optimizer used for training was the Adam optimizer with β1 = 0.9, β2 = 0.98 and ϵ = 10−9, as described in section 5.3 Optimizer on page 6.
📄 Sources: ['Page 7', 'Page 6']

❓ Your Question: exit

👋 Exiting QA System!


In [47]:

print("=" * 60)
print("PROJECT SUMMARY")
print("=" * 60)

print(f"""
Project: Document Question Answering System (RAG)
Document: Attention is All You Need (2017)

Tech Stack:
- LLM: Groq Llama 3.1 8B Instant
- Embeddings: sentence-transformers/all-MiniLM-L6-v2
- Vector DB: FAISS (saved permanently on Google Drive)
- Framework: LangChain
- Storage: Google Drive (permanent)

Pipeline:
1. Real PDF loaded from internet
2. Split into {len(chunks)} chunks
3. Converted to embeddings
4. Stored in FAISS Vector DB on Drive
5. RAG chain built with Groq LLM
6. Live Question Answering system

Evaluation Results:
- Total Questions: 3
- Correct: 3
- Accuracy: 100%

Key Features:
- Real research paper as document
- Permanent Vector DB on Google Drive
- Source page numbers shown with answers
- All QA logs saved to Drive
- Live interactive Q&A system
""")

print("=" * 60)
print("✅ Project Complete!")
print("=" * 60)

PROJECT SUMMARY

Project: Document Question Answering System (RAG)
Document: Attention is All You Need (2017)

Tech Stack:
- LLM: Groq Llama 3.1 8B Instant
- Embeddings: sentence-transformers/all-MiniLM-L6-v2
- Vector DB: FAISS (saved permanently on Google Drive)
- Framework: LangChain
- Storage: Google Drive (permanent)

Pipeline:
1. Real PDF loaded from internet
2. Split into 52 chunks
3. Converted to embeddings
4. Stored in FAISS Vector DB on Drive
5. RAG chain built with Groq LLM
6. Live Question Answering system

Evaluation Results:
- Total Questions: 3
- Correct: 3
- Accuracy: 100%

Key Features:
- Real research paper as document
- Permanent Vector DB on Google Drive  
- Source page numbers shown with answers
- All QA logs saved to Drive
- Live interactive Q&A system

✅ Project Complete!
